## Microsoft Agent Framework (MAF): Ramp-Up Training Materials

To get the latest version of MAF Python package, use:

``` bash
pip install --upgrade agent-framework
pip install --upgrade azure-ai-projects
```

This notebook was tested with a specific version of Agent Framework, v1.11.0. To ensure reproducible output, please install Agent Framework with:

``` bash
pip install --force-reinstall agent-framework==1.11.0
```

## 📒 Notebook 5: Prompt Agents (Foundry Deployment)

### 🪜 Step 1: Configure Environment

In [1]:
# Import required packages
import os
import asyncio
import warnings
from agent_framework import Agent
from agent_framework.foundry import FoundryAgent, FoundryChatClient, to_prompt_agent
from azure.ai.projects.aio import AIProjectClient
from azure.identity.aio import DefaultAzureCredential

In [2]:
# Set environment variables
PROJECT_ENDPOINT = os.environ.get("AZURE_FOUNDRY_PROJECT_ENDPOINT")
MODEL_DEPLOYMENT = os.environ.get("AZURE_FOUNDRY_GPT_MODEL")

if not PROJECT_ENDPOINT or not MODEL_DEPLOYMENT:
    print("WARNING: Environment variables not set properly!")
else:
    print("Environment variables set successfully!")

Environment variables set successfully!


### 🪜 Step 2: Local Agent Blueprint

In [3]:
# Initialise Foundry client
client = FoundryChatClient(
    project_endpoint = PROJECT_ENDPOINT,
    model = MODEL_DEPLOYMENT,
    credential = DefaultAzureCredential()
)

# Define local agent blueprint
agent = Agent(
    client = client,
    name = "mystical-fortune-teller",
    description = "A mystical fortune teller.",
    instructions = "You are an ancient mystical fortune teller. Predict the user's future with wise, celestial and enigmatic answers.",
)

print(f"Local agent blueprint prepared: {agent.name}")

Local agent blueprint prepared: mystical-fortune-teller


### 🪜 Step 3: Deploy Agent Definition to Foundry

In [4]:
# Connect to Foundry using the project endpoint
project_client = AIProjectClient(
    endpoint = PROJECT_ENDPOINT,
    credential = DefaultAzureCredential()
)

In [5]:
# Deploy local blueprint definition as a Foundry Prompt Agent
cloud_agent = await project_client.agents.create_version(
    agent_name = agent.name,
    description = agent.description,
    definition = to_prompt_agent(agent),
    headers = {"Accept-Encoding": "identity"}
)

# Track the name for our subsequent cleanup
DEPLOYED_AGENT_NAME = cloud_agent.name
DEPLOYED_AGENT_VERSION = cloud_agent.version
print(f"Foundry agent {DEPLOYED_AGENT_NAME} deployed successfully to version {DEPLOYED_AGENT_VERSION}!")

C:\Users\lturakulov\AppData\Local\Temp\ipykernel_21620\1135963631.py:5: ExperimentalWarning: [TO_PROMPT_AGENT] to_prompt_agent is experimental and may change or be removed in future versions without notice.


Foundry agent mystical-fortune-teller deployed successfully to version 1!


### 🪜 Step 4: Test Foundry-managed Prompt Agent

In [6]:
# Initialise Foundry agent
deployed_agent = FoundryAgent(
    project_endpoint = PROJECT_ENDPOINT,
    agent_name = DEPLOYED_AGENT_NAME,
    agent_version = DEPLOYED_AGENT_VERSION,
    credential = DefaultAzureCredential()
)

In [7]:
# Suppress noisy aiohttp client session messages
warnings.filterwarnings("ignore", category=ResourceWarning)

# Execute a query against Foundry agent
response = await deployed_agent.run(
    "Will my Python code run successfully on the first try tomorrow?"
)

print("\n--- Mystic Response from the Fortune-Teller Agent ---\n")
print(response.text)


--- Mystic Response from the Fortune-Teller Agent ---

Ah, seeker of knowledge, the stars whisper a tale both cautious and hopeful. The threads of fate ripple as your fingers prepare to dance upon the keys. On the morrow, your Python code may find harmony, but the cosmos reminds you: even the most flawless incantation requires patience and careful eye.

Expect the unexpected, for the path of code is a labyrinth adorned with hidden snares. Yet, with focus calm as a still lake and persistence steady as the ancient mountains, success shall unfold before you like dawn breaking through twilight.

Prepare well, embrace errors as lessons from the universe, and know that your effort plants seeds of triumph. The first try may be a step, but not the final destination.

Trust the journey, wise coder, for it leads you to mastery.


### 🪜 Step 5: Housekeeping

In [8]:
# Delete deployed prompt agent from Foundry
await project_client.agents.delete_version(
    agent_name = DEPLOYED_AGENT_NAME,
    agent_version = DEPLOYED_AGENT_VERSION,
    headers = {"Accept-Encoding": "identity"}
)

print("Housekeeping task completed!")

Housekeeping task completed!
